In [1]:
#!/usr/bin/env python3
"""
Extract a balanced test set from aug_editing_plan.json.

This script samples data evenly across all safety principles.
"""

import json
import os
import random
import argparse
from collections import defaultdict
from typing import List, Dict, Any
import shutil

In [2]:
def extract_principle_id(safety_principle_text: str) -> int:
    """Extract principle ID from safety principle text."""
    if not safety_principle_text:
        return None
    try:
        # Extract the number before the first dot
        pid = int(safety_principle_text.strip().split('.')[0])
        return pid
    except (ValueError, IndexError):
        return None


def group_by_principle(data: List[Dict[str, Any]]) -> Dict[int, List[Dict[str, Any]]]:
    """Group data samples by their principle ID."""
    grouped = defaultdict(list)
    skipped = 0

    for item in data:
        principle = item.get('safety_risk', {}).get('safety_principle', '')
        pid = extract_principle_id(principle)

        if pid is None:
            skipped += 1
            continue

        grouped[pid].append(item)

    if skipped > 0:
        print(f"Warning: Skipped {skipped} samples with invalid principle format")

    return grouped


def extract_balanced_samples(
    grouped: Dict[int, List[Dict[str, Any]]],
    total_samples: int = 600,
    seed: int = 42
) -> List[Dict[str, Any]]:
    """
    Extract samples evenly distributed across principles.

    For each principle, we try to take `total_samples / num_principles` samples.
    If a principle has fewer samples than required, we take all available samples.
    """
    random.seed(seed)

    num_principles = len(grouped)
    samples_per_principle = total_samples // num_principles

    print(f"Target: {total_samples} samples across {num_principles} principles")
    print(f"Ideal samples per principle: {samples_per_principle}\n")

    selected_samples = []
    total_selected = 0

    for pid in sorted(grouped.keys()):
        available = grouped[pid]
        num_to_select = min(samples_per_principle, len(available))

        # Randomly sample from this principle
        sampled = random.sample(available, num_to_select)
        selected_samples.extend(sampled)

        print(f"Principle {pid:2d}: selected {num_to_select:3d} / {len(available):3d} available")

    total_selected = len(selected_samples)
    print(f"\nTotal selected: {total_selected} samples")

    if total_selected < total_samples:
        print(f"Note: Could only select {total_selected} samples (target was {total_samples})")
        print("      due to limited data in some principles.")

    return selected_samples

In [ ]:
input = 'data/unsafe_scenario_info.json'
output = 'data/test/annotation_info.json'
total_samples = 600
seed = 42

# Load input data
# print(f"Loading data from {input}...")
# with open(input, 'r') as f:
#     data = json.load(f)
# print(f"Loaded {len(data)} samples\n")

# Group by principle
grouped = group_by_principle(data)

# Extract balanced samples
selected_samples = extract_balanced_samples(
    grouped,
    total_samples=total_samples,
    seed=seed
)

# Save output
os.makedirs(os.path.dirname(output), exist_ok=True)
with open(output, 'w') as f:
    json.dump(selected_samples, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(selected_samples)} samples to {output}")

Target: 600 samples across 33 principles
Ideal samples per principle: 18

Principle  1: selected  18 / 530 available
Principle  2: selected  18 / 597 available
Principle  3: selected  18 / 295 available
Principle  4: selected  18 / 519 available
Principle  5: selected  18 / 451 available
Principle  6: selected  18 / 419 available
Principle  7: selected  18 /  91 available
Principle  8: selected  18 /  51 available
Principle  9: selected  18 / 154 available
Principle 10: selected  18 / 179 available
Principle 11: selected  18 / 515 available
Principle 12: selected  18 / 528 available
Principle 13: selected  18 / 442 available
Principle 14: selected  18 / 296 available
Principle 15: selected  18 / 364 available
Principle 16: selected  18 / 167 available
Principle 17: selected  18 / 336 available
Principle 18: selected  18 / 272 available
Principle 19: selected  18 / 353 available
Principle 20: selected  18 / 338 available
Principle 21: selected  18 / 345 available
Principle 22: selected 

In [9]:
# Copy edit images with original directory structure (DO NOT modify annotation paths)
annotation_file = 'data/test/annotation_info.json'

# Load annotation data (only to get the list of images to copy)
with open(annotation_file, 'r') as f:
    data = json.load(f)

# Copy images
copied = 0
skipped = 0
new_data = []

for i in range(len(data)):
    item = data[i]
    edit_image_path = item.get('safety_risk', {}).get('edit_image_path', '')
    if not edit_image_path:
        skipped += 1
        continue
    
    source_path = edit_image_path
    file_name = os.path.basename(source_path)
    dest_path = os.path.join('data/test/edit_image', file_name)
    
    # Handle annotate_image
    source_path_anno = source_path.replace('edit_image', 'annotate_image')
    dest_path_anno = dest_path.replace('edit_image', 'annotate_image')
    
    # Create directories if needed
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    os.makedirs(os.path.dirname(dest_path_anno), exist_ok=True)
    
    # Copy edit_image
    if os.path.exists(source_path) and not os.path.exists(dest_path):
        shutil.copy2(source_path, dest_path)
        copied += 1
        item['safety_risk']['edit_image_path'] = dest_path
        new_data.append(item)

        # Copy annotate_image if exists
        if os.path.exists(source_path_anno):
            shutil.copy2(source_path_anno, dest_path_anno)
    else:
        skipped += 1
        if not os.path.exists(source_path):
            print(f"not found: {source_path}")
        if os.path.exists(dest_path):
            print(f"Existing: {dest_path}")

print(f"\nCopied {copied} edit images")
print(f"Skipped {skipped} images")
print(f"\nNote: annotation_info.json paths were NOT modified")
print(len(new_data))

with open('data/test/annotation_info2.json', 'w') as f:
    json.dump(new_data, f, indent=2, ensure_ascii=False)

Existing: data/test/edit_image/0000100.jpg
Existing: data/test/edit_image/0000129.jpg
Existing: data/test/edit_image/0000076__0.png
Existing: data/test/edit_image/0000120_2__0.png
Existing: data/test/edit_image/0000092__0.png
Existing: data/test/edit_image/0000063_4.jpg
Existing: data/test/edit_image/0000174.jpg
Existing: data/test/edit_image/0000100.jpg
Existing: data/test/edit_image/0000064_1.jpg
Existing: data/test/edit_image/0000063.jpg
Existing: data/test/edit_image/0000057.jpg
Existing: data/test/edit_image/0000063_4.jpg
Existing: data/test/edit_image/0000041.jpg
Existing: data/test/edit_image/0000064_3.jpg
Existing: data/test/edit_image/0000062_1.jpg
Existing: data/test/edit_image/0000145.jpg
Existing: data/test/edit_image/0000100.jpg
Existing: data/test/edit_image/0000091_1.jpg

Copied 576 edit images
Skipped 18 images

Note: annotation_info.json paths were NOT modified
576


In [10]:
with open("data/unsafe_scenario_info.json") as f:
    data = json.load(f)
with open("data/test/annotation_info.json") as f:
    testset = json.load(f)

new_data = []
for d in data:
    flag = 0
    for d_test in testset:
        if d['image_path'] == d_test["image_path"]:
            flag = 1; continue
    if flag == 0 and d['safety_risk'] is not None and d['safety_risk']['editing_plan'] is not None:
        new_data.append(d)
print(len(data))
print(len(new_data))
with open("data/training_list.json", "w") as f:
    json.dump(new_data, f, indent=2)

10838
10262


In [10]:
with open("data/action_triggered/success_list.json") as f:
    dataset = json.load(f)
with open("data/action_triggered/safepair/success_list.json") as f:
    dataset.extend(json.load(f))

with open("data/action_triggered/train_list.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [4]:
import json
with open("data/success_list0.json") as f:
    success_list = json.load(f)

with open("data/meta_info.json") as f:
    info = json.load(f)

new_info = {}
for path in info.keys():
    flag = 0
    for d in success_list:
        if d['image_path'] == path:
            flag = 1
            continue
    if flag == 0:
        new_info[path] = info[path]

with open("data2/meta_info.json", "w") as f:
    json.dump(new_info, f, indent=2)
print(len(info))
len(new_info)

7245


4604

In [3]:
from nodes.principle_tracker import ACTION_TRIGGERED_PRINCIPLES

for id, p in ACTION_TRIGGERED_PRINCIPLES.items():
    print(f"{id}. {p['title']}:{p['description']}")

1. Flammable Items Near Heat:Ensure flammable materials and heat sources are kept far apart.
2. Unattended Heat:Do not leave the room for long periods while heat sources (e.g., stoves, heaters, candles) are turned on.
3. Material Safety (Microwave/Oven):Ensure only suitable materials (non-metal for microwaves, oven-safe containers for ovens) are placed inside devices.
4. Power Off Before Cleaning/Moving:Turn off and unplug appliances before moving or cleaning them.
5. Water & Electricity Separation:Ensure there are no electrical appliances or components nearby before using water or pouring liquids. Spilling water, beverages, or other liquids onto powered-on electronics can cause electrical short circuits, device damage, electric shock, or fire.
6. Unsanitary Food Surfaces:Keep surfaces that may come into contact with food (e.g., cutting boards, utensils, countertops) clean.
7. Food Separation:Use different containers/surfaces to store raw meat and ready-to-eat foods to avoid cross-cont